# ml_E_unsup_case.ipynb

**所属章节**：Chapter E 无监督学习  
**主题**：① A 股行业聚类（基于收益率相关矩阵）② PCA 宏观因子提取  
**数据**：模拟 A 股 30 个行业的月度收益率（2000-2025）+ 30 个宏观指标  
**运行说明**：顺序执行，约 1 分钟


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,
    'axes.spines.top':False,'axes.spines.right':False})
C = {'primary':'#2166AC','secondary':'#D6604D','tertiary':'#4DAC26',
     'neutral':'#878787','highlight':'#B2182B','fill':'#D1E5F0'}
SEED=42; np.random.seed(SEED)
print('设置完成')

---
## 1. 数据生成

### 1a. A 股 30 个行业月度收益率（2000-2025）


In [ ]:
# 模拟 30 个行业的月度收益率（300 个月，约 25 年）
n_months, n_ind = 300, 30
industry_names = [
    '银行','保险','券商','信托','金融其他',
    '房地产','建筑建材','建筑装饰','建筑施工','物业管理',
    '钢铁','有色金属','煤炭','石油石化','化工',
    '电力','电气设备','光伏','风电','储能',
    '医药生物','医疗器械','医疗服务','中药','生物制品',
    '食品饮料','家用电器','纺织服装','汽车','消费电子'
]

# 5 个板块因子 + 行业特异性成分
sector_factor = np.random.randn(n_months, 5)
loadings = np.zeros((n_ind, 5))
loadings[ 0: 5, 0] = np.random.uniform(0.6,0.9,5)  # 金融板块
loadings[ 5:10, 1] = np.random.uniform(0.6,0.9,5)  # 地产建筑
loadings[10:15, 2] = np.random.uniform(0.6,0.9,5)  # 周期品
loadings[15:20, 3] = np.random.uniform(0.6,0.9,5)  # 新能源
loadings[20:25, 3] = np.random.uniform(0.4,0.7,5)  # 医药（与新能源适度相关）
loadings[25:30, 4] = np.random.uniform(0.6,0.9,5)  # 消费

# 市场因子（所有行业都暴露）
market_factor = np.random.randn(n_months, 1)
market_load   = np.random.uniform(0.3, 0.6, (n_ind, 1))

ret = (market_factor @ market_load.T
     + sector_factor @ loadings.T
     + 0.3 * np.random.randn(n_months, n_ind))

df_ret = pd.DataFrame(ret, columns=industry_names)
df_ret.index = pd.date_range('2000-01', periods=n_months, freq='ME')
print(f'行业收益率数据：{df_ret.shape}，时间范围：{df_ret.index[0].date()} 至 {df_ret.index[-1].date()}')

### 1b. 30 个宏观经济指标（月度）


In [ ]:
# 模拟宏观指标（GDP增速、CPI、M2、PMI 等 30 个指标）
macro_names = [
    'GDP增速','工业增加值','固定资产投资','消费增速','出口增速',
    'CPI','PPI','核心CPI','食品价格','能源价格',
    'M0','M1','M2','社会融资','信贷增速',
    '一年期LPR','十年国债利率','信用利差','汇率','外汇储备',
    '制造业PMI','非制造业PMI','综合PMI','工业企业利润','财政收入',
    '房价指数','股市成交量','北向资金','融资余额','VIX中国'
]
n_macro = 30

# 3 个潜在宏观因子 + 特异成分
latent = np.random.randn(n_months, 3)
M_load = np.random.randn(n_macro, 3) * 0.6
macro_data = latent @ M_load.T + 0.5*np.random.randn(n_months, n_macro)

df_macro = pd.DataFrame(macro_data, columns=macro_names)
df_macro.index = df_ret.index
print(f'宏观指标数据：{df_macro.shape}')

---
## 2. A 股行业聚类分析


In [ ]:
# 计算行业收益率相关矩阵
corr = df_ret.corr()
dist = 1 - corr   # 以 1-相关系数 作为距离
np.fill_diagonal(dist.values, 0)

# 层次聚类（Ward 方法）
Z = linkage(squareform(dist.values, checks=False), method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.subplots_adjust(wspace=0.35)

# 左图：相关矩阵热力图（按聚类排序）
agg = AgglomerativeClustering(n_clusters=5, linkage='ward')
cluster_labels = agg.fit_predict(dist)
order = np.argsort(cluster_labels)
corr_sorted = corr.iloc[order, order]

sns.heatmap(corr_sorted, ax=axes[0], cmap='RdBu_r',
            center=0, vmin=-0.5, vmax=1,
            xticklabels=corr_sorted.columns,
            yticklabels=corr_sorted.columns,
            linewidths=0.3, cbar_kws={'shrink':0.8})
axes[0].set_title('行业收益率相关矩阵\n（按聚类结果排序）', fontsize=12)
axes[0].tick_params(labelsize=7)

# 右图：树状图
dendrogram(Z, labels=industry_names, ax=axes[1],
           color_threshold=0.7*max(Z[:,2]),
           above_threshold_color='gray',
           leaf_font_size=8, leaf_rotation=45)
axes[1].axhline(0.7*max(Z[:,2]), color=C['highlight'],
                ls='--', lw=1.5, label='切割线（5个簇）')
axes[1].set_ylabel('Ward 距离')
axes[1].set_title('A 股行业层次聚类树状图', fontsize=12)
axes[1].legend(fontsize=9)

fig.suptitle('A 股行业聚类分析（基于月度收益率相关性，2000-2025）', fontsize=13)
fig.savefig('figs/fig_E_case_industry_cluster.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 输出聚类结果
print('\n行业聚类结果（5个簇）：')
for k in range(5):
    inds = [industry_names[i] for i in np.where(cluster_labels==k)[0]]
    print(f'  簇 {k+1}：{"，".join(inds)}')

---
## 3. PCA 宏观因子提取


In [ ]:
# 标准化宏观指标
sc_macro = StandardScaler()
macro_sc = sc_macro.fit_transform(df_macro)

# PCA：先提取 10 个主成分，看碎石图
pca_macro = PCA(n_components=10)
pca_macro.fit(macro_sc)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 左：碎石图
evr  = pca_macro.explained_variance_ratio_
cevr = evr.cumsum()
ax1  = axes[0]
ax2  = ax1.twinx()
ax1.bar(range(1,11), evr, color=C['primary'], alpha=0.75)
ax2.plot(range(1,11), cevr, 'o-', color=C['secondary'], lw=2, ms=6)
ax2.axhline(0.7, color=C['neutral'], ls='--', lw=1.2)
ax1.set_xlabel('主成分编号')
ax1.set_ylabel('方差解释比（EVR）', color=C['primary'])
ax2.set_ylabel('累积 EVR', color=C['secondary'])
ax2.set_ylim(0,1.05)
ax1.set_title('宏观指标碎石图（PCA）')
ax1.spines['top'].set_visible(False); ax2.spines['top'].set_visible(False)

# 右：前3个主成分的因子载荷热力图（取绝对值前10个变量）
n_pc = 3
loadings_df = pd.DataFrame(
    pca_macro.components_[:n_pc].T,
    index=macro_names,
    columns=[f'PC{i+1}' for i in range(n_pc)]
)
# 按 PC1 载荷绝对值排序，取前12个
top12 = loadings_df['PC1'].abs().nlargest(12).index
sns.heatmap(loadings_df.loc[top12], ax=axes[1],
            cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            cbar_kws={'shrink':0.8}, linewidths=0.3,
            annot_kws={'size':8})
axes[1].set_title(f'前 {n_pc} 个主成分的因子载荷\n（按 PC1 绝对载荷排序，前12个指标）')
axes[1].tick_params(labelsize=8)

fig.tight_layout()
fig.savefig('figs/fig_E_case_pca_macro.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print(f'\n前3个主成分的方差解释比：')
for i in range(3):
    print(f'  PC{i+1}: {evr[i]:.1%}  （累积: {cevr[i]:.1%}）')

---
## 4. 主成分因子得分的时序图


In [ ]:
# 提取前3个主成分得分（时间序列）
pca3 = PCA(n_components=3)
pc_scores = pca3.fit_transform(macro_sc)
df_pc = pd.DataFrame(pc_scores, index=df_macro.index,
                      columns=['宏观因子1','宏观因子2','宏观因子3'])

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
fig.subplots_adjust(hspace=0.25)
colors_ts = [C['primary'], C['secondary'], C['tertiary']]

for i, (col, color) in enumerate(zip(df_pc.columns, colors_ts)):
    ax = axes[i]
    ax.plot(df_pc.index, df_pc[col], color=color, lw=1.2)
    ax.axhline(0, color='k', lw=0.7, alpha=0.4)
    ax.fill_between(df_pc.index, df_pc[col], 0,
                    where=df_pc[col]>0, alpha=0.2, color=color)
    ax.fill_between(df_pc.index, df_pc[col], 0,
                    where=df_pc[col]<0, alpha=0.2, color=C['neutral'])
    evr_i = pca3.explained_variance_ratio_[i]
    ax.set_ylabel(f'{col}\n(EVR={evr_i:.1%})', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_title('宏观因子得分时序图（PCA 前3个主成分，2000-2025）')
axes[-1].set_xlabel('日期')
fig.savefig('figs/fig_E_case_pc_timeseries.png',
            dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('宏观因子得分时序图已保存')

---
## 提示词模板 #8：PCA 降维

```
背景：对 p 列宏观指标做 PCA 降维，提取主成分因子。

我的数据：df（DataFrame，n 行，p 列数值特征，已去除缺失值）

请帮我：
1. StandardScaler 标准化
2. 拟合 PCA（n_components=min(10,p)）
3. 绘制碎石图（双轴：EVR 柱状图 + 累积 EVR 折线 + 80% 阈值线）
4. 打印前5个主成分的 EVR 和累积 EVR
5. 输出前3个主成分的因子载荷热力图（heatmap）
6. 用前 K 个主成分变换数据，保存为 df_pca（列名 PC1...PCK）
7. 所有代码中文注释
```

## 提示词模板 #9：K-means 聚类

```
背景：对金融数据做 K-means 聚类。

我的数据：X_scaled（标准化特征矩阵），labels_list（样本名称列表）

请帮我：
1. K=2 到 K=10，分别计算 WCSS 和轮廓系数
2. 绘制双图（肘部法则 + 轮廓系数），标注建议 K
3. 用建议 K 重新拟合（n_init=10，random_state=42）
4. 输出每个簇的样本名称列表
5. 绘制前两个 PCA 成分空间的聚类散点图（含质心标注）
6. 所有代码中文注释
```


In [ ]:
print('='*50)
print('Chapter E 案例分析完成')
print('='*50)
print('\n案例1：A 股行业聚类')
print(f'  数据：{n_ind} 个行业，{n_months} 个月')
print(f'  方法：Ward 层次聚类（基于 1-相关系数 距离）')
print(f'  结果：5 个行业板块（见上方聚类结果）')
print('\n案例2：PCA 宏观因子提取')
print(f'  数据：{n_macro} 个宏观指标，{n_months} 个月')
print(f'  前3个主成分累积解释方差：{cevr[2]:.1%}')
print('\n输出文件：')
print('  figs/fig_E_case_industry_cluster.png')
print('  figs/fig_E_case_pca_macro.png')
print('  figs/fig_E_case_pc_timeseries.png')